# 9.6 Recursive filters

Every filter we have seen so far computes its output purely from the _input_. But there is no reason a difference equation cannot also refer to _past outputs_. Filters that do are called {vocab}`recursive filters`, and they open up a large and powerful new class of behaviors.

## Signal-flow diagrams

Recursive filters are often best understood visually, as a {vocab}`signal-flow diagram`. This is yet another perspective on filters, complementing difference equations, convolution, and impulse responses. The diagrams are built from three elements: wires that carry a signal, a summing junction (drawn as a circled plus) that adds signals together, and a _delay block_ labeled $z^{-1}$, which delays its input by exactly one sample, mapping $x[n]$ to $x[n-1]$.

:::{margin}
The notation $z^{-1}$ comes from the _z-transform_, a generalization of the DFT that is the standard tool for analyzing recursive filters. We won't cover the z-transform in this course, but we will still adopt the conventional $z^{-N}$ notation in signal flow diagrams for a delay of $N$ sample. Note that delaying by one sample is itself just convolution with the impulse response $\color{red}{h} = [0, 1]$.
:::

:::{figure}
![Two signal-flow diagrams side by side. Left, labeled feedforward only: the input x[n] splits, one path going straight to a summing junction and another passing through a z-to-the-minus-one delay block before reaching the junction, whose output is y[n]; the equation is y[n] = x[n] + x[n-1]. Right, labeled feedback only: the input x[n] goes straight to a summing junction whose output y[n] is also tapped and fed back through a z-to-the-minus-one block into the junction; the equation is y[n] = x[n] + y[n-1].](./assets/fig-recursive-signalflow.png)

Left: a _feedforward_ filter, $y[n] = x[n] + x[n-1]$, whose output depends only on the input (a delayed copy of the _input_ is added in). Right: a _feedback_ filter, $y[n] = x[n] + y[n-1]$, whose output depends on itself (a delayed copy of the _output_ is added in). The feedback loop is what makes a filter recursive.
:::

The left diagram is an ordinary filter: the input flows forward through a delay and a sum to the output. The right diagram adds a _feedback_ loop, "tapping" the output, delaying it, and feeding it back into the sum. This feedback is what makes the filter recursive. Contrasting the two side by side, the only difference is whether the delayed copy fed into the sum comes from the input ($x[n-1]$, feedforward) or from the output ($y[n-1]$, feedback).

## Generalized difference equation

We can generalize the difference equation to include both past inputs and past outputs.

:::{prf:definition} General recursive difference equation
:label: def-recursive
A recursive filter is defined by

$$
\begin{aligned}
\purple{y[n]} = {}& b_0\,\blue{x[n]} + b_1\,\blue{x[n-1]} + \cdots + b_M\,\blue{x[n-M]} && \text{(feedforward)} \\
& \phantom{b_0\,\blue{x[n]}}{} + a_1\,\purple{y[n-1]} + \cdots + a_L\,\purple{y[n-L]} && \text{(feedback)}
\end{aligned}
$$

The $M{+}1$ {vocab}`feedforward coefficients` $b_i$ act on past _inputs_, and the $L$ {vocab}`feedback coefficients` $a_j$ act on past _outputs_. The feedforward part is exactly a convolution: the $b_i$ are the same numbers we called the impulse response $\red{h}$ earlier, just renamed $b$ by convention in the recursive setting. Note there is no $a_0$ term, since $y[n]$ cannot depend on _itself_, only on past outputs. The largest input and output delays are $M$ and $L$.
:::

Two facts about this general form are worth committing to memory. First, **every filter of this form is LTI**, feedback and all. Second, the {vocab}`order` of the filter is the largest delay it uses, $\max(M, L)$. For example, $y[n] = x[n] + x[n-1] + \tfrac{1}{3}y[n-2]$ has $M = 1$ and $L = 2$, so it is a second-order filter.

Recursive filters are commonplace in computer music because they can achieve higher-quality frequency responses with very few coefficients (and thus very little computation) compared to the equivalent non-recursive filter. More on "higher-quality" frequency responses in the next section!

## Finite and infinite impulse responses

Feedback has a striking consequence for the impulse response. Consider the simplest recursive filter,

$$y[n] = x[n] + y[n-1].$$

What is its response to the unit impulse $\delta = [1, 0, 0, \ldots]$? We can read the output off the difference equation one sample at a time, recalling that $y[n] = 0$ for $n < 0$:

$$
\begin{aligned}
y[0] &= x[0] + y[-1] &= 1 + 0 = 1, \\
y[1] &= x[1] + y[0]  &= 0 + 1 = 1, \\
y[2] &= x[2] + y[1]  &= 0 + 1 = 1, \\
y[3] &= x[3] + y[2]  &= 0 + 1 = 1, \\
     &\;\;\vdots
\end{aligned}
$$

The input contributes only its single initial $1$, but the feedback keeps copying the previous output forward forever. The impulse response is $[1, 1, 1, 1, \ldots]$, _infinitely long_. This particular filter accumulates a running sum of its input.

This distinguishes two families of filters:

1. A filter with only feedforward coefficients has a {vocab}`finite impulse response` (FIR) equal to the coefficients themselves. Its impulse response has as many nonzero samples as it has coefficients, and then stops.
1. A recursive filter (with feedback) generally has an {vocab}`infinite impulse response` (IIR). The feedback keeps the response going forever.

With feedback comes a new danger: an IIR filter can be {vocab}`unstable`. Compare two filters. The filter $y[n] = x[n] + 0.9\,y[n-1]$ is _stable_: each pass through the loop shrinks the signal by a factor of $0.9$, so its impulse response $[1, 0.9, 0.81, \ldots]$ decays toward zero. But $y[n] = x[n] + 1.1\,y[n-1]$ is _unstable_: each pass _amplifies_ the signal by $1.1$, so its impulse response $[1, 1.1, 1.21, \ldots]$ grows without bound and quickly explodes into a deafening blowup. Designing stable recursive filters is a central concern of filter design.

For implementation, an IIR filter must generally be run as a difference equation, computing each output from previous outputs, rather than as a direct convolution (its impulse response is infinite, so we cannot convolve with all of it). That said, the impulse response of a _stable_ IIR filter decays, so in practice we can approximate it by a finite one: run the impulse response until it has decayed below some threshold (say $60$ dB down, $|y[n]| \le 0.001$), truncate it there, and convolve with the result. The example below computes the impulse response of the stable recursive filter $y[n] = x[n] + 0.95\,y[n-100]$ and finds where it crosses the $-60$ dB truncation threshold:

In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# The stable recursive filter y[n] = x[n] + 0.95*y[n-100]. Its impulse response
# is a train of "echoes" 100 samples apart, each 0.95x the previous one, and it
# decays forever. To implement a stable IIR filter as a finite convolution, we
# truncate its impulse response once it falls below a threshold, here -60 dB.
a, delay, N = 0.95, 100, 20000
x = np.zeros(N)
x[0] = 1.0                                   # unit impulse
h = np.zeros(N)                              # the impulse response we build up
for n in range(N):
    h[n] = x[n] + a * (h[n - delay] if n >= delay else 0.0)

# The response peaks once per echo, at multiples of the 100-sample delay.
peak_n = np.arange(0, N, delay)
peak_db = 20 * np.log10(np.abs(h[peak_n]) / np.abs(h[0]))
cutoff = peak_n[np.argmax(peak_db < -60)]    # first echo below -60 dB

plt.figure(figsize=(10, 4))
plt.plot(peak_n, peak_db, color="C4", marker="o", markersize=3,
         label="impulse response peaks")
plt.axhline(-60, color="C3", linestyle="--", label="-60 dB threshold")
plt.axvline(cutoff, color="0.4", linestyle=":", label=f"truncate at n = {cutoff}")
plt.ylim(-100, 5)
plt.xlabel("Sample index n")
plt.ylabel("Peak amplitude (dB)")
plt.legend()
plt.show()

print(f"The impulse response decays below -60 dB at sample {cutoff} "
      f"(about {cutoff / 44100 * 1000:.0f} ms at 44.1 kHz), where we can truncate.")